In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams.update(
    {
        "text.usetex": False,
        "axes.labelsize": 20,
        "figure.labelsize": 18,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.constrained_layout.wspace": 0,
        "figure.constrained_layout.hspace": 0,
        "figure.constrained_layout.h_pad": 0,
        "figure.constrained_layout.w_pad": 0,
        "axes.linewidth": 1.2,
    }
)

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

## Kernel Approximations for Convolved Transfer Functions

This notebook collects the approximation experiments that are separate from the main transfer-function tutorial in `04_TransferFunctions.ipynb`.

The goal is twofold:

1. compare dense convolved kernels against a quasiseparable `SHOSeries` surrogate, and
2. fit a dense `ConvolvedKernel` directly to a simulated light curve using the direct solver.

The final section fits the physical kernel parameters themselves, without any series approximation in the likelihood.


In [ ]:
from scipy.optimize import minimize, nnls

from eztaox.kernels.eqx_utils import find_param_by_name
from eztaox.kernels.quasisep import Exp, Quasisep, SHO
from eztaox.kernels.transfer_function import (
    ConvolvedKernel,
    CausalGaussianTransferFunction,
    TransferFunction,
)
from eztaox.models import UniVarModel
from eztaox.simulator import UniVarSim
from eztaox.ts_utils import add_noise

In [ ]:
class CausalTopHatTransferFunction(TransferFunction):
    """A normalized causal top-hat transfer function."""

    def evaluate(self, X1, X2):
        dt = X2 - X1 - self.shift
        half_width = self.width / 2.0
        inside = (dt >= -half_width) & (dt <= half_width)
        return jnp.where(inside, 1.0 / jnp.maximum(self.width, 1e-12), 0.0)

### 1. Dense convolved kernels

We will compare approximations for two transfer functions applied to the same DRW/OU base kernel:

- a causal Gaussian
- a causal top-hat


In [ ]:
base_kernel = Exp(scale=80.0, sigma=0.2)
gaussian_tf = CausalGaussianTransferFunction(width=80.0, shift=30.0)
tophat_tf = CausalTopHatTransferFunction(width=80.0, shift=30.0)

convolved_kernel = ConvolvedKernel(
    base_kernel=base_kernel,
    transfer_function=gaussian_tf,
    n_grid=2048,
)
convolved_kernel_tophat = ConvolvedKernel(
    base_kernel=base_kernel,
    transfer_function=tophat_tf,
    n_grid=2048,
)

tau_grid = jnp.linspace(0.0, 300.0, 600)
k_base = jax.vmap(lambda tau: base_kernel.evaluate(0.0, tau))(tau_grid)
k_conv = jax.vmap(lambda tau: convolved_kernel.evaluate(0.0, tau))(tau_grid)
k_conv_tophat = jax.vmap(lambda tau: convolved_kernel_tophat.evaluate(0.0, tau))(
    tau_grid
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(np.asarray(tau_grid), np.asarray(k_base), label="Base Exp", lw=2)
axes[0].plot(np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian direct", lw=2)
axes[0].plot(
    np.asarray(tau_grid), np.asarray(k_conv_tophat), label="Top-hat direct", lw=2
)
axes[0].set_xlabel(r"Lag $|\Delta t|$")
axes[0].set_ylabel(r"$k(\Delta t)$")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian direct", lw=2)
axes[1].plot(
    np.asarray(tau_grid), np.asarray(k_conv_tophat), label="Top-hat direct", lw=2
)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel(r"$k(\Delta t)$")
axes[1].set_title("Dense target kernels")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

In [ ]:
def stable_sho_cov(tau, omega, q, sigma=1.0):
    tau = jnp.asarray(tau)
    if q < 0.5:
        f = jnp.sqrt(jnp.maximum(1.0 - 4.0 * q**2, 1e-12))
        lam_slow = omega * (1.0 - f) / (2.0 * q)
        lam_fast = omega * (1.0 + f) / (2.0 * q)
        w_slow = 0.5 * (1.0 + 1.0 / f)
        w_fast = 0.5 * (1.0 - 1.0 / f)
        return sigma**2 * (
            w_slow * jnp.exp(-lam_slow * tau) + w_fast * jnp.exp(-lam_fast * tau)
        )
    atom = SHO(omega=omega, quality=q, sigma=sigma)
    return atom.evaluate(0.0, tau)


class SHOSeries(Quasisep):
    """Nonnegative mixture of fixed SHO atoms fit in the time domain."""

    omegas: jax.Array
    qualities: jax.Array
    weights: jax.Array

    @classmethod
    def from_kernel(
        cls,
        kernel,
        n_omega: int = 10,
        q_grid=(0.15, 0.25, 0.4),
        n_fit: int = 512,
        tau_max: float | None = None,
        scale_min_ratio: float = 0.03,
        scale_max_ratio: float = 30.0,
        weight_floor_ratio: float = 1e-3,
        zero_lag_boost: float = 20.0,
    ):
        kernel_scales = find_param_by_name(kernel, "scale")
        if kernel_scales is None:
            raise ValueError("Kernel must have a 'scale' parameter for auto placement.")

        base_scale = float(sum(kernel_scales) / len(kernel_scales))
        if tau_max is None:
            tau_max = scale_max_ratio * base_scale

        tau_min = max(scale_min_ratio * base_scale * 1e-2, 1e-6)
        tau_fit = np.concatenate([[0.0], np.geomspace(tau_min, tau_max, n_fit - 1)])
        tau_basis = np.geomspace(
            max(scale_min_ratio * base_scale, 1e-6),
            max(scale_max_ratio * base_scale, scale_min_ratio * base_scale * 1.01),
            n_omega,
        )
        omegas = 1.0 / tau_basis
        qualities = np.asarray(q_grid, dtype=float)

        tau_fit_jax = jnp.asarray(tau_fit)
        k_fit = np.asarray(
            jax.vmap(lambda tau: kernel.evaluate(jnp.array(0.0), tau))(tau_fit_jax)
        )

        atoms = [(omega, q) for q in qualities for omega in omegas]
        A = np.column_stack(
            [
                np.asarray(stable_sho_cov(tau_fit_jax, omega, q, sigma=1.0))
                for omega, q in atoms
            ]
        )

        k0 = float(k_fit[0])
        sigma_w = np.maximum(np.abs(k_fit), weight_floor_ratio * max(k0, 1e-12))
        sigma_w[0] /= zero_lag_boost
        Aw = A / sigma_w[:, None]
        bw = k_fit / sigma_w
        weights, _ = nnls(Aw, bw)

        wsum = weights.sum()
        if wsum > 0:
            weights *= k0 / wsum

        omega_arr = np.asarray([omega for omega, _ in atoms], dtype=float)
        q_arr = np.asarray([q for _, q in atoms], dtype=float)
        return cls(
            omegas=jnp.asarray(omega_arr),
            qualities=jnp.asarray(q_arr),
            weights=jnp.asarray(weights),
        )

    def _atoms(self):
        return [
            SHO(omega=self.omegas[i], quality=self.qualities[i], sigma=1.0)
            for i in range(self.omegas.shape[0])
        ]

    def coord_to_sortable(self, X):
        return X

    def design_matrix(self):
        atoms = self._atoms()
        return jax.scipy.linalg.block_diag(*[atom.design_matrix() for atom in atoms])

    def stationary_covariance(self):
        atoms = self._atoms()
        return jax.scipy.linalg.block_diag(
            *[
                self.weights[i] * atom.stationary_covariance()
                for i, atom in enumerate(atoms)
            ]
        )

    def observation_model(self, X):
        atoms = self._atoms()
        return jnp.concatenate([atom.observation_model(X) for atom in atoms])

    def transition_matrix(self, X1, X2):
        atoms = self._atoms()
        return jax.scipy.linalg.block_diag(
            *[atom.transition_matrix(X1, X2) for atom in atoms]
        )

    def power(self, f, df=None):
        del df
        out = 0.0
        for i, atom in enumerate(self._atoms()):
            out = out + self.weights[i] * atom.power(f)
        return out

In [ ]:
def kernel_params(kernel):
    return {
        "scale": float(kernel.base_kernel.scale),
        "sigma": float(kernel.base_kernel.sigma),
        "width": float(kernel.transfer_function.width),
        "shift": float(kernel.transfer_function.shift),
    }

### 2. Gaussian transfer function: compare the dense kernel and `SHOSeries`

We fit the `SHOSeries` approximation only on the lag range shown in the plot. That avoids spending fit capacity on very long lags that are irrelevant for this comparison and also avoids numerical overflow in the overdamped SHO basis.


In [ ]:
sho_series = SHOSeries.from_kernel(
    convolved_kernel,
    n_omega=18,
    q_grid=(0.08, 0.12, 0.18, 0.25, 0.35, 0.49),
    n_fit=1024,
    tau_max=250.0,
    zero_lag_boost=50.0,
)

k_sho = jax.vmap(lambda tau: sho_series.evaluate(0.0, tau))(tau_grid)

In [ ]:
frac_err_sho = (k_sho - k_conv) / jnp.maximum(jnp.abs(k_conv), 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian direct", lw=2)
axes[0].plot(
    np.asarray(tau_grid), np.asarray(k_sho), label="Gaussian SHOSeries", lw=2, ls=":"
)
axes[0].set_xlabel(r"Lag $|\Delta t|$")
axes[0].set_ylabel(r"$k(\Delta t)$")
axes[0].set_title("Gaussian kernel comparison")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(np.asarray(tau_grid), np.asarray(frac_err_sho), label="SHOSeries", lw=2)
axes[1].axhline(0.0, color="0.2", lw=1)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel("Fractional error")
axes[1].set_title("Gaussian approximation error")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

### 3. Top-hat transfer function: compare the dense kernel and `SHOSeries`

The top-hat case is harder because of the sharper structure introduced by the transfer function. The same `SHOSeries` workflow applies.


In [ ]:
sho_series_tophat = SHOSeries.from_kernel(
    convolved_kernel_tophat,
    n_omega=10,
    q_grid=(0.08, 0.12, 0.18, 0.25, 0.35, 0.49),
    n_fit=512,
    tau_max=float(tau_grid.max()),
)

k_sho_tophat = jax.vmap(lambda tau: sho_series_tophat.evaluate(0.0, tau))(tau_grid)

In [ ]:
frac_err_sho_tophat = (k_sho_tophat - k_conv_tophat) / jnp.maximum(
    jnp.abs(k_conv_tophat), 1e-12
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
axes[0].plot(
    np.asarray(tau_grid), np.asarray(k_conv_tophat), label="Top-hat direct", lw=2
)
axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(k_sho_tophat),
    label="Top-hat SHOSeries",
    lw=2,
    ls=":",
)
axes[0].set_xlabel(r"Lag $|\Delta t|$")
axes[0].set_ylabel(r"$k(\Delta t)$")
axes[0].set_title("Top-hat kernel comparison")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(
    np.asarray(tau_grid), np.asarray(frac_err_sho_tophat), label="SHOSeries", lw=2
)
axes[1].axhline(0.0, color="0.2", lw=1)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel("Fractional error")
axes[1].set_title("Top-hat approximation error")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

### 4. Fit a dense `ConvolvedKernel` directly to a simulated light curve

Finally, we fit the physical parameters of the dense `ConvolvedKernel` itself using the direct solver. This avoids any series approximation in the likelihood: the fitted parameters are the base-kernel scale and amplitude together with the transfer-function width and shift.


In [ ]:
# Use a smaller FFT grid in the direct-fit demo to keep the likelihood evaluations reasonable.
true_fit_kernel = ConvolvedKernel(
    base_kernel=Exp(scale=80.0, sigma=0.2),
    transfer_function=CausalGaussianTransferFunction(width=80.0, shift=30.0),
    n_grid=512,
)
fit_kernel0 = ConvolvedKernel(
    base_kernel=Exp(scale=55.0, sigma=0.12),
    transfer_function=CausalGaussianTransferFunction(width=55.0, shift=18.0),
    n_grid=512,
)

flat_true, _ = jax.flatten_util.ravel_pytree(true_fit_kernel)
flat0, convolved_unravel = jax.flatten_util.ravel_pytree(fit_kernel0)

sim_params = {"log_kernel_param": jnp.log(flat_true)}
sim = UniVarSim(true_fit_kernel, 1.0, 300.0, sim_params, zero_mean=True)
t_obs, y_true = sim.random(48, jax.random.PRNGKey(11), jax.random.PRNGKey(12))
yerr = jnp.full_like(t_obs, 0.02)
y_obs = add_noise(y_true, yerr, jax.random.PRNGKey(13))

model = UniVarModel(t_obs, y_obs, yerr, fit_kernel0, zero_mean=True)
fit_params0 = {"log_kernel_param": jnp.log(jnp.clip(flat0, 1e-12, None))}

print("Initial log probability:", float(model.log_prob(fit_params0)))
print("True kernel parameters:", kernel_params(true_fit_kernel))
print("Initial kernel parameters:", kernel_params(fit_kernel0))

In [ ]:
def objective(log_param_np):
    x = jnp.asarray(log_param_np)
    lp = model.log_prob({"log_kernel_param": x})
    if not jnp.isfinite(lp):
        return 1e30
    return float(-lp)


rng = np.random.default_rng(14)
x0_center = np.asarray(fit_params0["log_kernel_param"])
bounds = [(x - 1.0, x + 1.0) for x in x0_center]

best_res = None
candidate_starts = [x0_center] + [
    x0_center + 0.15 * rng.normal(size=x0_center.shape) for _ in range(8)
]
for x0 in candidate_starts:
    res = minimize(objective, x0=x0, method="L-BFGS-B", bounds=bounds)
    if best_res is None or res.fun < best_res.fun:
        best_res = res

if best_res is None or not np.isfinite(best_res.fun) or best_res.fun >= 1e29:
    bestP = fit_params0
else:
    bestP = {"log_kernel_param": jnp.asarray(best_res.x)}

print("Best log probability:", float(model.log_prob(bestP)))

In [ ]:
t_pred = jnp.linspace(0.0, 300.0, 400)
mu_pred, std_pred = model.pred(bestP, t_pred)
best_kernel = convolved_unravel(jnp.exp(bestP["log_kernel_param"]))

k_direct_true = jax.vmap(lambda tau: true_fit_kernel.evaluate(0.0, tau))(tau_grid)
k_direct_init = jax.vmap(lambda tau: fit_kernel0.evaluate(0.0, tau))(tau_grid)
k_direct_best = jax.vmap(lambda tau: best_kernel.evaluate(0.0, tau))(tau_grid)

kernel_rmse_init = float(jnp.sqrt(jnp.mean((k_direct_init - k_direct_true) ** 2)))
kernel_rmse_best = float(jnp.sqrt(jnp.mean((k_direct_best - k_direct_true) ** 2)))
print("Kernel RMSE (initial -> target):", kernel_rmse_init)
print("Kernel RMSE (best fit -> target):", kernel_rmse_best)
print("Recovered kernel parameters:", kernel_params(best_kernel))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

axes[0].fill_between(
    np.asarray(t_pred),
    np.asarray(mu_pred - std_pred),
    np.asarray(mu_pred + std_pred),
    color="tab:blue",
    alpha=0.2,
)
axes[0].plot(
    np.asarray(t_pred),
    np.asarray(mu_pred),
    color="tab:blue",
    lw=2,
    label="Direct ConvolvedKernel fit",
)
axes[0].errorbar(
    np.asarray(t_obs),
    np.asarray(y_obs),
    np.asarray(yerr),
    fmt=".",
    color="0.2",
    label="Mock data",
)
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Flux")
axes[0].set_title("Light-curve fit")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(
    np.asarray(tau_grid), np.asarray(k_direct_true), lw=2, label="Dense target kernel"
)
axes[1].plot(
    np.asarray(tau_grid),
    np.asarray(k_direct_init),
    lw=2,
    ls="--",
    label="Initial guess",
)
axes[1].plot(
    np.asarray(tau_grid),
    np.asarray(k_direct_best),
    lw=2,
    ls=":",
    label="Best direct fit",
)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel(r"$k(\Delta t)$")
axes[1].set_title("Recovered covariance")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

### Notes

- The comparison sections in this notebook now focus only on `SHOSeries`; the nonnegative exponential mixture has been removed.
- In the final section, the likelihood uses the dense `ConvolvedKernel` directly, so the optimized parameters are the physical base-kernel and transfer-function parameters rather than surrogate-series coefficients.
- This direct fit is much slower than the quasiseparable surrogate fits because each likelihood evaluation uses the dense GP solver and the FFT-based convolved kernel.
- For that reason, the direct-fit demo uses fewer data points and a smaller FFT grid than the approximation-comparison sections.
